In [ ]:
import sys
import os

sys.path.append(os.path.abspath("."))

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader
from data.dataset import FlickrDataset, CollateWrapper
from models.captioner import ImageCaptioner
from utils.visualization import Visualizer

def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Гиперпараметры конфигурации
    embed_size = 512
    num_heads = 8
    num_layers = 4
    forward_expansion = 4
    dropout = 0.1
    batch_size = 64
    epochs = 5
    lr = 3e-4

    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    ])

    # Инициализация путей датасета Flickr8k
    dataset = FlickrDataset(root_dir="data/Images", captions_file="data/captions.txt", transform=transform)
    pad_idx = dataset.vocab.stoi["<PAD>"]

    loader = DataLoader(
        dataset=dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, collate_fn=CollateWrapper(pad_idx)
    )

    model = ImageCaptioner(
        embed_size, len(dataset.vocab), num_heads, num_layers, forward_expansion, dropout
    ).to(device)

    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    loss_history = []
    model.train()

    print("Старт обучения...")
    for epoch in range(epochs):
        for idx, (images, captions) in enumerate(loader):
            images, captions = images.to(device), captions.to(device)

            outputs = model(images, captions[:, :-1])
            loss = criterion(outputs.reshape(-1, outputs.shape[-1]), captions[:, 1:].reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if idx % 100 == 0:
                print(f"Эпоха [{epoch+1}/{epochs}], Шаг [{idx}/{len(loader)}], Ошибка: {loss.item():.4f}")
                loss_history.append(loss.item())

    # Сохраняем веса модели
    torch.save(model.state_dict(), "resnet_transformer_captioner.pth")
    print("Обучение завершено. Модель сохранена.")

    # Визуализируем график ошибок
    Visualizer.plot_losses(loss_history)

if __name__ == "__main__":
    run_training()


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\user/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100.0%


Старт обучения...
Эпоха [1/5], Шаг [0/633], Ошибка: 8.2165
